## The Transformation Logic

In [0]:
query = """
WITH order_payments_agg AS (
    -- Aggregate payments per order to avoid duplicating rows during joins
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value,
        MAX(payment_installments) AS max_payment_installments
    FROM
        silver.crm_payments
    GROUP BY
        order_id
),
order_items_agg AS (
    -- Aggregate item counts and costs per order
    SELECT
        order_id,
        COUNT(product_id) AS total_items_ordered,
        SUM(price) AS total_item_price,
        SUM(shipping_charges) AS total_shipping_charges
    FROM
        silver.crm_order_items
    GROUP BY
        order_id
)
SELECT
    o.order_id,
    o.customer_id,
    c.zip_code_prefix AS customer_zip_code,
    c.state_code AS customer_state,
    o.order_status,
    
    -- Timestamps & Dates
    o.purchase_timestamp,
    o.approved_at,
    o.delivered_timestamp,
    o.estimated_delivery_date,
    
    -- Calculated Operational Metrics
    DATEDIFF(o.delivered_timestamp, o.purchase_timestamp) AS days_to_deliver,
    DATEDIFF(o.estimated_delivery_date, o.delivered_timestamp) AS days_ahead_of_schedule,
    
    -- Financial & Quantity Metrics
    COALESCE(i.total_items_ordered, 0) AS total_items_ordered,
    COALESCE(i.total_item_price, 0.0) AS total_item_price,
    COALESCE(i.total_shipping_charges, 0.0) AS total_shipping_charges,
    COALESCE(p.total_payment_value, 0.0) AS total_payment_value,
    COALESCE(p.max_payment_installments, 1) AS max_payment_installments

FROM
    silver.crm_orders o
LEFT JOIN 
    silver.crm_customers c ON o.customer_id = c.customer_id
LEFT JOIN 
    order_items_agg i ON o.order_id = i.order_id
LEFT JOIN 
    order_payments_agg p ON o.order_id = p.order_id;
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

## Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_fact_sales")

### Sanity checks of Gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_fact_order_items LIMIT 10